# Real-estate and roll-forward statement templates

`finstack_quant.statements_analytics` ships reusable schedule builders that attach nodes onto a `finstack_quant.statements` model. This notebook covers:

- **Property operating statement** -- a rent roll rolling up to EGI, NOI, and NCF, with free rent, occupancy, growth, and a management fee.
- **NOI / NCF buildups** -- lighter-weight revenue / expense / capex aggregations.
- **Vintage (cohort) buildup** -- a decay-curve convolution over a new-volume driver.
- **Roll-forward schedule** -- beginning / ending balances from increases and decreases.
- **Corkscrew articulation check** -- a runtime validation that balance movements reconcile.

### Workflow

Each `add_*` template takes a model (or its JSON) and **returns a JSON spec string**. Rebuild a `FinancialModelSpec` from it, then evaluate:

`ModelBuilder` -> `add_*` (returns JSON) -> `FinancialModelSpec.from_json` -> `Evaluator().evaluate` -> `result.get(node, period)`

## Imports

In [1]:
import json

import pandas as pd

from finstack_quant import statements
from finstack_quant.statements_analytics import (
    LeaseSpec,
    ManagementFeeSpec,
    ManagementFeeBase,
    PropertyTemplateNodes,
    add_property_operating_statement,
    add_noi_buildup,
    add_ncf_buildup,
    add_vintage_buildup,
    add_roll_forward_with_opening,
    AccountType,
    CorkscrewAccount,
    CorkscrewConfig,
    CorkscrewExtension,
)

PERIODS = ["2025Q1", "2025Q2", "2025Q3", "2025Q4"]

## Property operating statement (rent roll -> EGI -> NOI -> NCF)

We model two leases over four quarters. `lease_a` starts in Q1 with one free-rent period and 1%/period escalation at 95% occupancy; `lease_b` starts in Q2 at 90% occupancy. Other income, operating expenses, and capex are supplied as explicit driver nodes, and a 5% management fee is charged on EGI.

In [2]:
b = statements.ModelBuilder("office_property")
b.periods("2025Q1..Q4", None)
b.value("parking_income", [(p, 10.0) for p in PERIODS])
b.value("property_taxes", [(p, 5.0) for p in PERIODS])
b.value("repairs", [(p, 2.0) for p in PERIODS])
b.value("capex", [(p, 3.0) for p in PERIODS])

leases = [
    LeaseSpec(
        node_id="lease_a", start="2025Q1", base_rent=100.0, end="2025Q4",
        free_rent_periods=1, occupancy=0.95, growth_rate=0.01,
    ),
    LeaseSpec(
        node_id="lease_b", start="2025Q2", base_rent=60.0, end="2025Q4",
        occupancy=0.90,
    ),
]

spec_json = add_property_operating_statement(
    b.build(),
    leases=leases,
    other_income_nodes=["parking_income"],
    opex_nodes=["property_taxes", "repairs"],
    capex_nodes=["capex"],
    management_fee=ManagementFeeSpec(rate=0.05, base=ManagementFeeBase.Egi),
    nodes=PropertyTemplateNodes(),
)
result = statements.Evaluator().evaluate(statements.FinancialModelSpec.from_json(spec_json))

stack = ["rent_pgi", "free_rent", "vacancy_loss", "rent_effective", "other_income_total",
         "egi", "management_fee", "opex_total", "noi", "capex_total", "ncf"]
pd.DataFrame({p: {n: result.get(n, p) for n in stack} for p in PERIODS})

,2025Q1,2025Q2,2025Q3,2025Q4
rent_pgi,100.0,161.0000,162.010000,163.030100
free_rent,100.0,0.0000,0.000000,0.000000
vacancy_loss,0.0,11.0500,11.100500,11.151505
rent_effective,0.0,149.9500,150.909500,151.878595
other_income_total,10.0,10.0000,10.000000,10.000000
egi,10.0,159.9500,160.909500,161.878595
management_fee,0.5,7.9975,8.045475,8.093930
opex_total,7.5,14.9975,15.045475,15.093930
noi,2.5,144.9525,145.864025,146.784665
capex_total,3.0,3.0000,3.000000,3.000000


Note Q1: the free-rent period zeroes effective rent while opex and the management fee still apply, so NOI is negative -- a realistic lease-up dynamic. Per-lease effective rent is also exposed:

In [3]:
pd.DataFrame({
    p: {f"{lid}.effective_rent": result.get(f"{lid}.effective_rent", p) for lid in ("lease_a", "lease_b")}
    for p in PERIODS
})

,2025Q1,2025Q2,2025Q3,2025Q4
lease_a.effective_rent,0.0,95.95,96.9095,97.878595
lease_b.effective_rent,0.0,54.00,54.0000,54.000000


## NOI / NCF buildups (lighter weight)

When you already have revenue and expense driver nodes, `add_noi_buildup` aggregates them into totals and a NOI node (`NOI = total revenue - total expenses`), and `add_ncf_buildup` subtracts capex (`NCF = NOI - capex`).

In [4]:
nb = statements.ModelBuilder("noi_ncf")
nb.periods("2025Q1..Q2", None)
nb.value("rent", [("2025Q1", 100.0), ("2025Q2", 110.0)])
nb.value("other_income", [("2025Q1", 10.0), ("2025Q2", 12.0)])
nb.value("taxes", [("2025Q1", 20.0), ("2025Q2", 22.0)])
nb.value("repairs", [("2025Q1", 5.0), ("2025Q2", 6.0)])
nb.value("capex", [("2025Q1", 3.0), ("2025Q2", 4.0)])

j = add_noi_buildup(nb.build(), "total_revenue", ["rent", "other_income"], "total_expenses", ["taxes", "repairs"], "noi")
j = add_ncf_buildup(j, "noi", ["capex"], "ncf")
res = statements.Evaluator().evaluate(statements.FinancialModelSpec.from_json(j))

pd.DataFrame({p: {n: res.get(n, p) for n in ("total_revenue", "total_expenses", "noi", "ncf")} for p in ("2025Q1", "2025Q2")})

,2025Q1,2025Q2
total_revenue,110.0,122.0
total_expenses,25.0,28.0
noi,85.0,94.0
ncf,82.0,90.0


## Vintage (cohort) buildup

`add_vintage_buildup` convolves a new-volume driver with a decay curve. Each period's value is `sum(new_volume[t-k] * decay[k])` -- useful for cohort revenue, loan balances, or any origination-and-runoff pattern. Here a decay curve of `[1.0, 0.8, 0.5, 0.0]` means a cohort contributes fully on origination, then 80%, 50%, and 0%.

In [5]:
vb = statements.ModelBuilder("cohort_revenue")
vb.periods("2025Q1..2025Q4", None)
vb.value("new_sales", [("2025Q1", 100.0), ("2025Q2", 200.0), ("2025Q3", 300.0), ("2025Q4", 400.0)])

decay = [1.0, 0.8, 0.5, 0.0]
vj = add_vintage_buildup(vb.build(), "revenue", "new_sales", decay)
vres = statements.Evaluator().evaluate(statements.FinancialModelSpec.from_json(vj))

pd.DataFrame({p: {"new_sales": vres.get("new_sales", p), "revenue": vres.get("revenue", p)} for p in PERIODS})

,2025Q1,2025Q2,2025Q3,2025Q4
new_sales,100.0,200.0,300.0,400.0
revenue,100.0,280.0,510.0,740.0


## Roll-forward schedule (beginning / ending balances)

`add_roll_forward_with_opening` creates `{name}_beg` and `{name}_end` nodes where `end = beg + sum(increases) - sum(decreases)` and the next period's beginning equals the prior period's end. Increases and decreases are supplied as **positive magnitudes** (the template subtracts the decreases).

In [6]:
rf = statements.ModelBuilder("debt_balance")
rf.periods("2024Q1..2024Q4", None)
rf.value("draws", [("2024Q1", 10.0), ("2024Q2", 15.0), ("2024Q3", 0.0), ("2024Q4", 5.0)])
rf.value("repayments", [("2024Q1", 5.0), ("2024Q2", 5.0), ("2024Q3", 5.0), ("2024Q4", 5.0)])

rj = add_roll_forward_with_opening(rf.build().to_json(), "balance", ["draws"], ["repayments"], 100.0)
rres = statements.Evaluator().evaluate(statements.FinancialModelSpec.from_json(rj))

debt_periods = ["2024Q1", "2024Q2", "2024Q3", "2024Q4"]
pd.DataFrame({p: {"balance_beg": rres.get("balance_beg", p), "balance_end": rres.get("balance_end", p)} for p in debt_periods})

,2024Q1,2024Q2,2024Q3,2024Q4
balance_beg,100.0,105.0,115.0,110.0
balance_end,105.0,115.0,110.0,110.0


## Corkscrew articulation check

A corkscrew validates that the change nodes feeding a balance actually reconcile its period-over-period movement. Unlike the roll-forward template, corkscrew **change nodes are additive**, so an outflow is entered as a negative number. Build and evaluate the model first, then run the extension over the results.

In [7]:
bs = statements.ModelBuilder("balance_sheet")
bs.periods("2025Q1..2025Q3", None)
bs.value("cash", [("2025Q1", 100_000.0), ("2025Q2", 110_000.0), ("2025Q3", 115_000.0)])
bs.value("cash_inflows", [("2025Q1", 0.0), ("2025Q2", 15_000.0), ("2025Q3", 10_000.0)])
bs.value("cash_outflows", [("2025Q1", 0.0), ("2025Q2", -5_000.0), ("2025Q3", -5_000.0)])

bs_model = bs.build()
bs_results = statements.Evaluator().evaluate(bs_model)

config = CorkscrewConfig(
    accounts=[
        CorkscrewAccount(node_id="cash", account_type=AccountType.Asset, changes=["cash_inflows", "cash_outflows"]),
    ],
    tolerance=0.01,
    fail_on_error=False,
)
report = CorkscrewExtension.with_config(config).execute(bs_model, bs_results)

print(f"corkscrew status: {report.status}")
pd.DataFrame(json.loads(report.data_json())["validations"])

corkscrew status: success


,account,is_valid,max_error,periods_validated,type
0,cash,True,0.0,2,Asset


## Takeaways

- Templates attach nodes onto a `statements` model and return a JSON spec; rebuild with `FinancialModelSpec.from_json(...)` before evaluating.
- `add_property_operating_statement` turns a list of `LeaseSpec`s plus driver nodes into a full rent roll -> EGI -> NOI -> NCF stack (free rent, occupancy, growth, and management fee included); `add_noi_buildup` / `add_ncf_buildup` are lighter aggregations.
- `add_vintage_buildup` convolves a volume driver with a decay curve for cohort / runoff patterns.
- `add_roll_forward_with_opening` builds beginning / ending balance nodes from positive-magnitude increases and decreases.
- `CorkscrewExtension` is a runtime check that change nodes (entered additively) reconcile a balance's movement within tolerance.